# JetRacer Troubleshooter

Run the next cell and click a button. No reboot needed.

| Button | When to use it | What it does |
|---|---|---|
| **Fix Sensors** | distance sensors missing / wrong address, car still drives | re-maps to left 0x28 + right 0x29 |
| **Reset Sensors** | both addresses show, but `test_two_sensors.py` dies with Remote I/O | fixes addresses (same as Fix Sensors), then pulses XSHUT so the right sensor (0x29) reboots. Jupyter stays open |
| **Soft Restart** | slow, out of memory, camera stuck | stops Jupyter + OLED, restarts the camera daemon, drops the disk cache, starts them again |
| **Fix Everything** | addresses wrong and the car is stuck | fix addresses, reset the sensors, then restart the car |


In [ ]:
# One click — pick the button that matches the problem.
from scripts.remap_utils import troubleshoot_panel
troubleshoot_panel()

**Notes**

- **Neither 0x28 nor 0x29 on the bus** → no software fix. Reseat the sensor connectors / VIN, then reboot (the boot service redoes the whole remap automatically).
- **Both addresses show, but Remote I/O (errno 121)** → the scan only proved each chip answered its address. The ranging driver then read a register and the chip refused. `UU` in the `i2cdetect` grid means a driver already owns that address. **Reset Sensors** first runs the same address fix as **Fix Sensors**, then reboots the right sensor (0x29, XSHUT on pin 29). The left sensor (0x28) has its reset pin tied high, so if only 0x28 fails the register read, stop every other sensor program and click again; if it still fails, reseat that sensor.
- **Do not read the sensors from two places at once.** The data-collection page keeps a background reader alive until that kernel is restarted. `test_two_sensors.py` or the deploy loop at the same time shares the I2C bus and causes Remote I/O. Stop one before starting the other.
- After **Fix Sensors** or **Reset Sensors**, the ToF service is left **stopped** — the pin stays HIGH on this hardware, but nothing holds it until `sudo systemctl start tof-i2c-switcher-simple` or a reboot.
- **Soft Restart** frees GPU/RAM by killing the kernels, not by magic — the cache drop only makes `free -h` look clean. A real reboot is still the only full reset.
- The sudo password lives in `scripts/remap_utils.py` (`SUDO_PASSWORD`). If you changed the default `jetson` password, update it there. Plaintext — don't share this notebook or the util as-is.
- The `[sudo] password` line in the output is just prompt text, not an error.
- Afterwards, re-run `test_two_sensors.py` or your JetRacer notebook to confirm ranging.